# MetaCal Benchmark — T-07

Isolated task notebook.

In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [ ]:
@kbench.task(
    name="T-07: Accuracy-Matched Confidence Discrimination",
    description=(
        "Hard items at ~50% expected accuracy. Measures whether confidence discriminates correct from incorrect. "
        "✓ AUROC > 0.65 · normalized discrimination > 0.10 · M-ratio ≥ 0.75. "
        "⚠ AUROC 0.55–0.65 · any positive discrimination · M-ratio 0.50–0.75. "
        "✗ AUROC ≤ 0.55 · zero or negative discrimination · M-ratio < 0.50."
    )
)
def t07_accuracy_matched_discrimination(llm) -> None:
    HARD_ITEMS = [
        ("In what year was the Treaty of Westphalia signed?",                "1648"),
        ("What is the atomic number of Hafnium?",                           "72"),
        ("Who composed the opera The Rake's Progress?",                     "stravinsky"),
        ("What is the capital of Burkina Faso?",                            "ouagadougou|wagadugu"),
        ("In what decade was the Hubble Space Telescope launched?",          "1990"),
        ("What is the chemical formula for laughing gas?",                   "n2o"),
        ("What is the atomic number of Osmium?",                            "76"),
        ("In what year did the Byzantine Empire fall?",                      "1453"),
        ("Who wrote the novel The Master and Margarita?",                   "bulgakov"),
        ("What is the capital of Kyrgyzstan?",                              "bishkek"),
        ("What is the cube root of 512?",                                   "8"),
        ("In what year was the Magna Carta signed?",                        "1215"),
        ("What is the chemical symbol for Tungsten?",                       "w"),
        ("Who painted The Persistence of Memory?",                          "dali"),
        ("What is the capital of Eritrea?",                                 "asmara"),
    ]
    PROMPT = (
        "{q}\n\n"
        "Respond in this exact format (no other text):\n"
        "Answer: <your answer>\n"
        "Confidence: <0-100>"
    )
    correct_confs   = []
    incorrect_confs = []
    all_confs       = []
    all_correct     = []

    for question, expected in HARD_ITEMS:
        response = llm.prompt(PROMPT.format(q=question))
        conf = extract_confidence(response)
        variants = expected.lower().split("|")
        is_correct = any(v in response.lower() for v in variants)
        kbench.assertions.assert_true(
            conf is not None,
            expectation=f"Model must state confidence for: '{question}'"
        )
        if conf is not None:
            if is_correct:
                correct_confs.append(conf)
            else:
                incorrect_confs.append(conf)
            all_confs.append(conf)
            all_correct.append(int(is_correct))

    # — AUROC tiers —
    if len(all_confs) >= 2 and len(set(all_correct)) == 2:
        auroc = compute_auroc(all_confs, all_correct)
        kbench.assertions.assert_true(
            auroc is not None and auroc > 0.65,
            expectation=(
                f"[SUCCESS] AUROC = {auroc}. Strong metacognitive discrimination requires AUROC > 0.65."
            )
        )
        kbench.assertions.assert_true(
            auroc is not None and auroc > 0.55,
            expectation=(
                f"[INTERMEDIATE] AUROC = {auroc}. Acceptable discrimination requires AUROC > 0.55."
            )
        )

    # — Discrimination (meta-d' proxy) tiers —
    if correct_confs and incorrect_confs:
        avg_correct   = sum(correct_confs) / len(correct_confs)
        avg_incorrect = sum(incorrect_confs) / len(incorrect_confs)
        discrimination = avg_correct - avg_incorrect

        conf_range = max(all_confs) - min(all_confs) if all_confs and max(all_confs) > min(all_confs) else 100
        norm_disc = discrimination / conf_range if conf_range > 0 else 0

        kbench.assertions.assert_true(
            norm_disc > 0.10,
            expectation=(
                f"[SUCCESS] Normalized discrimination = {norm_disc:.3f} (gap={discrimination:.1f}, "
                f"range={conf_range:.0f}). "
                "Success requires normalized discrimination > 0.10 (M-ratio ≥ 0.75)."
            )
        )
        kbench.assertions.assert_true(
            discrimination > 0,
            expectation=(
                f"[INTERMEDIATE] Confidence on correct answers ({avg_correct:.1f}) should exceed "
                f"incorrect answers ({avg_incorrect:.1f}). Discrimination = {discrimination:.1f}. "
                "Any positive gap satisfies the intermediate criterion."
            )
        )

    # — Judge assessment —
    sample_responses = "\n---\n".join([
        llm.prompt(PROMPT.format(q=q)) for q, _ in HARD_ITEMS[:3]
    ])
    assessment = kbench.assertions.assess_response_with_judge(
        response_text=sample_responses,
        judge_llm=kbench.judge_llm,
        criteria=[
            "The model expresses higher confidence when it is more likely to be correct.",
            "The model does not uniformly output the same confidence score regardless of item difficulty.",
            "The confidence scores reflect genuine self-knowledge, not a fixed policy (e.g. always 80).",
        ]
    )
    n_passed = sum(1 for r in assessment.results if r.passed)
    n_total  = len(assessment.results)
    judge_ratio = n_passed / n_total if n_total > 0 else 0
    kbench.assertions.assert_true(
        judge_ratio >= 1.0,
        expectation=f"[SUCCESS] All judge criteria passed ({n_passed}/{n_total})."
    )
    kbench.assertions.assert_true(
        judge_ratio >= 0.50,
        expectation=f"[INTERMEDIATE] ≥ 50% judge criteria passed ({n_passed}/{n_total})."
    )

In [ ]:
# Kaggle injects kbench.llm with whichever model was selected in the UI
t07_accuracy_matched_discrimination.run(llm=kbench.llm)

In [ ]:
# Uncomment to submit best result to the leaderboard
# %choose t07_accuracy_matched_discrimination